# Sampling timing parameters with Discovery and Enterprise

This notebook demonstrates how `nltiming` would be used on an everyday basis: bind a `TimingSpec` to a pulsar, sample with Discovery (NumPyro) or Enterprise (PTMCMC), select backends, and plot the chain.

`decentered_model` is the everyday Discovery sampler model; notebook 3 explains the alternative. Charts and geometry wait until notebooks 2 and 4.

A `TimingPulsar` is required: an object that implements the necessary attributes (a protocol), just like the `PintPulsar` and `TempoPulsar` classes from `enterprise` do. The `TimingPulsar` is intentionally not derived from `BasePulsar`, but it works the same way. Today the implementation is comes from MetaPulsar; the `nltiming` calls do not change if Discovery or Enterprise develop a native implementation (which we hope they will).

In [ ]:
import os
import sys

# Timing residuals need float64. Set this before JAX/NumPyro import.
os.environ.setdefault("JAX_ENABLE_X64", "1")

import jax
import numpy as np
import matplotlib.pyplot as plt
import corner
import discovery as ds
from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, load_run
import nltiming.sampling as nlts  # nlts.numpyro / nlts.ptmcmc — not the sampler packages

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="WARNING")

# Re-assert x64 after nltiming/JAX have been imported (no-op if already on).
nlts.numpyro.ensure_x64()


## A TimingPulsar

AEI-DR2 combined J1721-2457 (isolated, 150 TOAs). Run this notebook from `examples/notebooks/`.

`create_metapulsar` builds the host object nltiming binds to. One PTA key (`combined`) with one par/tim pair is enough; the factory still goes through the MetaPulsar path so the same API works for multi-PTA objects later.


In [ ]:
DATA = Path("..") / "data" / "J1721-2457"
pulsar = create_metapulsar(
    # PTA name -> list of legs. "combined" is the AEI-DR2 merged release.
    {"combined": [{
        "par": DATA / "J1721-2457.par",
        "tim": DATA / "J1721-2457.tim",
        "timing_package": "tempo2",  # par/tim compatibility mode (tempo2 or pint)
    }]},
    combination_strategy="per_pta",  # keep this PTA's params; no cross-PTA merge
    use_pulse_numbers="reuse",       # honor -pn already on the TOAs (do not rederive)
)
print(pulsar.name, len(pulsar.toas))


## Discovery

Default inference samples the nonlinear axes (here proper motion) and analytically marginalizes the rest. White noise is the same EFAC-only model as Enterprise, using Discovery's default `Uniform(0.1, 10)`. Short pedagogical chain — scale `num_warmup` / `num_samples` for science.

The likelihood is three pieces stacked by Discovery: the residual vector, a diagonal EFAC noise kernel, and the timing delay signals from `timing.discovery_signals()`.


In [ ]:
# engines="jug" selects the JUG backend for this tempo2-family host.
# name="timing" is the signal stem (Enterprise/Discovery site prefix).
spec = TimingSpec(engines="jug", name="timing")
timing = spec.for_pulsar(pulsar)  # TimingSignal: sampled/marginalized plan + delays
print("sampled:", timing.sampled)
print("marginalized:", timing.marginalized)

# Create the discovery likelihood
efac = f"{pulsar.name}_efac"
EFAC_PRIOR = (0.1, 10.0)  # Discovery default: "(.*_)?efac" in discovery.prior.priordict_standard
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,  # data vector
    ds.makenoise_measurement_simple(pulsar, add_equad=False),  # N = EFAC^2 * sigma^2
    *timing.discovery_signals(),  # nonlinear timing delays for the sampled axes
])

# Create the Discovery model (numpyro-based)
# Everyday nltiming model: sample the nonlinear block, marginalize the rest
# inside likelihood.logL. Notebook 3 shows joint_model (inference="all").
model = nlts.numpyro.decentered_model(
    likelihood, timing, priors={efac: EFAC_PRIOR},
)

# Optional starts, not a full parameter list.
#
# `init_to_value` is NumPyro's initializer: sites in `values` are pinned,
# everything else falls back to a uniform draw from its prior. nltiming does
# not wrap that helper.
#
# `decentered_init_values` pins only `xi` to 0 (the transport origin → the
# par-file / conditional-center timing values, not physical zeros). We also
# pin EFAC to 1 as a typical white-noise start; omitting it is fine.
#
# Passing `init_strategy` replaces nlts.numpyro.nuts()'s default (joint-model
# timing zeros). For this decentered model we therefore pass xi ourselves.
from numpyro.infer import init_to_value
init = {
    **nlts.numpyro.decentered_init_values(timing, model.transport),
    efac: 1.0,
}
mcmc = nlts.numpyro.nuts(
    model, timing,
    num_warmup=200, num_samples=1000, num_chains=1,
    init_strategy=init_to_value(values=init),
)
mcmc.run(jax.random.PRNGKey(0))


## Physical posterior

`nlts.numpyro.posterior()` decodes the recorded timing offsets (`…_delta` deterministics) into chain-preserving ArviZ data with short parameter names and display units. Do not plot the latent `xi` site: that is a whitened coordinate, not a physical parameter.


In [ ]:
post = nlts.numpyro.posterior(mcmc, timing)
# var_names pins the panel order to the timing plan (ArviZ otherwise sorts A–Z).
corner.corner(post, var_names=list(timing.sampled))
plt.show()


## Enterprise

Same default inference plan, libstempo backend. Enterprise samples prior-normal `z` (`…_timing_PMRA`), not physical units; `load_run` / `run.posterior()` decode those to the same display names Discovery uses.

Raw `PTSampler` — no `enterprise_extensions`.


In [ ]:
import tempfile
from enterprise.signals import parameter, signal_base, white_signals
from PTMCMCSampler.PTMCMCSampler import PTSampler

# Same TimingSpec, different engine: libstempo instead of JUG.
spec_ent = spec.with_engines({"tempo2": "libstempo"})
timing_ent = spec_ent.for_pulsar(pulsar)
white = white_signals.MeasurementNoise(efac=parameter.Uniform(*EFAC_PRIOR))
# enterprise_signal() is the Enterprise twin of timing.discovery_signals().
pta = signal_base.PTA([(white + spec_ent.enterprise_signal())(pulsar)])
print(pta.param_names)

# Sidecar: space + chain column layout so load_run() can decode z → physical.
outdir = Path(tempfile.mkdtemp(prefix="nlt_ent_"))
timing_ent.write(
    outdir, likelihood="enterprise", sampler="ptmcmc",
    chain_layout=nlts.ptmcmc.chain_layout(timing_ent, pta.param_names),
)

x0 = np.hstack([np.asarray(p.sample(), dtype=float).reshape(-1) for p in pta.params])
sampler = PTSampler(
    len(pta.param_names), pta.get_lnlikelihood, pta.get_lnprior,
    np.diag(np.full(len(pta.param_names), 0.1**2)), outDir=str(outdir),
)
sampler.sample(x0, Niter=5000)



In [ ]:
run = load_run(outdir)  # reads the sidecar written above + chain_1.txt
post = run.posterior(burn=0.25)  # drop the first 25%; keys are timing.sampled names
corner.corner(
    np.column_stack([post[k] for k in timing_ent.sampled]),
    labels=list(timing_ent.sampled),
)
plt.show()


In [ ]:
# Overlay: same physical axes, two independent samplers.
# Rebuild the Discovery posterior here — the Enterprise cell reused `post`.
names = list(timing_ent.sampled)
ent = np.column_stack([post[k] for k in names])
disc = np.column_stack([
    np.asarray(nlts.numpyro.posterior(mcmc, timing).posterior[k]).reshape(-1)
    for k in names
])
# density=True + 1/N weights: 1D panels share a unit-area scale despite N_ent != N_disc.
hist_kw = {"density": True, "histtype": "step"}
fig = corner.corner(
    ent, labels=names, color="C0",
    weights=np.full(len(ent), 1.0 / len(ent)),
    hist_kwargs={**hist_kw, "color": "C0"},
)
corner.corner(
    disc, fig=fig, color="C1",
    plot_datapoints=False, plot_density=False,
    fill_contours=False, no_fill_contours=True,
    weights=np.full(len(disc), 1.0 / len(disc)),
    hist_kwargs={**hist_kw, "color": "C1"},
)
fig.legend(
    handles=[
        plt.Line2D([0], [0], color="C0", label="Enterprise"),
        plt.Line2D([0], [0], color="C1", label="Discovery"),
    ],
    loc="upper right",
)
plt.show()


In [ ]:
# EFAC is a white-noise hyperparameter, not a timing axis, so it is not in
# timing.sampled / run.posterior(). Read it from each sampler's raw chain.
disc_efac = np.asarray(mcmc.get_samples()[efac]).reshape(-1)
chain = np.loadtxt(outdir / "chain_1.txt")
ent_efac = chain[int(0.25 * len(chain)) :, list(pta.param_names).index(efac)]
plt.hist(ent_efac, bins=40, density=True, histtype="step", color="C0", label="Enterprise")
plt.hist(disc_efac, bins=40, density=True, histtype="step", color="C1", label="Discovery")
plt.xlabel(efac)
plt.ylabel("density")
plt.legend()


In [ ]:
from enterprise.signals import signal_base

In [ ]:
from nltiming import TimingSpec
from enterprise.signals import parameter, signal_base, white_signals

EFAC_PRIOR = (0.1, 10.0)  # Discovery default

# Create `nltiming` TimingSignal
spec = TimingSpec(engines="jug", name="timing")
timing = spec.for_pulsar(pulsar)  # TimingSignal: sampled/marginalized plan + delays

# Create the discovery likelihood
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar),
    *timing.discovery_signals(),  # Instead of the usual ds.makegp_timing(psr),
])

# Create the Enterprise likelihood
white = white_signals.MeasurementNoise(efac=parameter.Uniform(*EFAC_PRIOR))
pta = signal_base.PTA([(white + spec.enterprise_signal())(pulsar)])